Once the model is trained, it is needed to choose the best performing one:
1. finding the best head/layer combination
2. finding the best training step

For this, we'll use the validation step that is already mapped

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# imports
import json
import pandas as pd
from rxnmapper.find_best_head_layer import FindBestHeadLayerCombination

/home/yves/anaconda3/envs/rxnmapper_v2/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-03-25 16:46:44.451912: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-03-25 16:46:44.453449: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2025-03-25 16:46:44.485281: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow wit

In [3]:
# validation set unmapped reactions
val_set = pd.read_json('../data/RXNMapperData/Validation/eval_schneider_filtered.json')
val_rxns = val_set['rxn'].tolist()
val_ref_rxns = val_set['rxn_atom_mapped'].tolist()

## 1. finding the best head/layer combination

All the heads will be tested on the three different layers (there are 8 heads, 12 layers)

In [4]:
fbh = FindBestHeadLayerCombination(
    rxns_list=val_rxns, 
    ref_rxns_list=val_ref_rxns, 
    heads = range(8), 
    layers = [9, 10, 11], 
    model_path='../rxnmapper/models/transformers/albert_heads_8_uspto_all_1310k/', 
    model_name='original_IBM', 
)
dict_to_eval = fbh.get_config_dict_to_eval()


Model config validated.


Layers: 100%|██████████| 3/3 [00:00<00:00, 349.69it/s]


In [5]:
predictions_df = fbh.map_rxns_from_models_in_dict()

Models: 100%|██████████| 24/24 [02:04<00:00,  5.17s/it]


In [6]:
eval_result_dict = fbh.evaluate_results()

100%|██████████| 969/969 [00:04<00:00, 209.40it/s]

Best head/layer combination for model albert_original_IBM_head_5_layer_10, getting 950 matches out of 969 --> 98.04% accuracy


## 2. finding the best training step